# Anomaly Detection Model - Bank Transaction Fraud Risk

Notebook ini membangun model anomaly detection untuk mengidentifikasi transaksi yang berpotensi fraud. Pendekatan yang digunakan adalah unsupervised learning karena dataset tidak memiliki label fraud eksplisit.

Model yang digunakan:
- Isolation Forest (primary)
- Local Outlier Factor / LOF (pembanding)

Evaluasi dilakukan secara kualitatif dengan menganalisis karakteristik transaksi yang ditandai sebagai anomali.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
df = pd.read_csv('../data/bank_transactions_data_2.csv')
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])
print(f'Loaded {len(df)} transactions')

## 1. Feature Engineering

Dari hasil EDA, beberapa fitur yang relevan untuk deteksi anomali:

- **AmountToBalanceRatio**: rasio amount terhadap saldo, transaksi yang terlalu besar relatif terhadap saldo patut dicurigai
- **TxnHour**: jam transaksi, transaksi dini hari bisa jadi indikator
- **DaysSincePrevTxn**: jarak waktu dari transaksi sebelumnya, nilai negatif atau sangat kecil bisa anomali
- **LoginAttempts**: jumlah percobaan login, nilai tinggi bisa indikasi brute force
- **TransactionDuration**: durasi transaksi yang terlalu cepat atau terlalu lama
- **Channel & TransactionType**: encoded sebagai numerik

In [ ]:
# temporal features
df['TxnHour'] = df['TransactionDate'].dt.hour
df['TxnDayOfWeek'] = df['TransactionDate'].dt.dayofweek
df['DaysSincePrevTxn'] = (df['TransactionDate'] - df['PreviousTransactionDate']).dt.total_seconds() / 86400

# ratio features
df['AmountToBalanceRatio'] = df['TransactionAmount'] / (df['AccountBalance'] + 1)

# flag: apakah transaksi terjadi di jam tidak wajar (00:00 - 05:59)
df['IsLateNight'] = (df['TxnHour'].between(0, 5)).astype(int)

# encode categorical
le_channel = LabelEncoder()
le_txntype = LabelEncoder()
le_occupation = LabelEncoder()

df['ChannelEncoded'] = le_channel.fit_transform(df['Channel'])
df['TxnTypeEncoded'] = le_txntype.fit_transform(df['TransactionType'])
df['OccupationEncoded'] = le_occupation.fit_transform(df['CustomerOccupation'])

print('Features created')
df[['TxnHour', 'DaysSincePrevTxn', 'AmountToBalanceRatio', 'IsLateNight', 'ChannelEncoded']].describe()

In [ ]:
# pilih fitur untuk model
feature_cols = [
    'TransactionAmount',
    'TransactionDuration',
    'LoginAttempts',
    'AccountBalance',
    'CustomerAge',
    'TxnHour',
    'TxnDayOfWeek',
    'DaysSincePrevTxn',
    'AmountToBalanceRatio',
    'IsLateNight',
    'ChannelEncoded',
    'TxnTypeEncoded',
    'OccupationEncoded'
]

X = df[feature_cols].copy()

# handle missing/inf values
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)

print(f'Feature matrix shape: {X.shape}')
X.head()

In [ ]:
# scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print('Scaling done')
X_scaled.describe().round(2)

## 2. Isolation Forest

Isolation Forest bekerja dengan prinsip bahwa anomali lebih mudah "diisolasi" dibanding data normal. Data point yang membutuhkan lebih sedikit split untuk diisolasi dianggap lebih anomalous.

Parameter:
- `contamination=0.05`: asumsi awal sekitar 5% transaksi berpotensi anomali
- `n_estimators=200`: jumlah tree yang cukup untuk stabilitas
- `random_state=42`: reproducibility

In [ ]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

df['IsoForest_Pred'] = iso_forest.fit_predict(X_scaled)
df['IsoForest_Score'] = iso_forest.decision_function(X_scaled)

# -1 = anomaly, 1 = normal
n_anomalies = (df['IsoForest_Pred'] == -1).sum()
print(f'Isolation Forest detected {n_anomalies} anomalies ({n_anomalies/len(df)*100:.1f}%)')

In [ ]:
# distribusi anomaly score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['IsoForest_Score'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=df[df['IsoForest_Pred'] == -1]['IsoForest_Score'].max(), color='red', linestyle='--', label='Threshold')
axes[0].set_title('Distribusi Anomaly Score (Isolation Forest)')
axes[0].set_xlabel('Anomaly Score')
axes[0].legend()

colors = ['steelblue' if x == 1 else 'red' for x in df['IsoForest_Pred']]
axes[1].scatter(df['TransactionAmount'], df['AccountBalance'], c=colors, alpha=0.5, s=15)
axes[1].set_xlabel('Transaction Amount')
axes[1].set_ylabel('Account Balance')
axes[1].set_title('Anomalies: Amount vs Balance (merah = anomali)')

plt.tight_layout()
plt.show()

## 3. Local Outlier Factor (LOF)

LOF mengukur seberapa "terisolasi" suatu data point dibanding tetangganya. Nilai LOF yang tinggi menandakan data point tersebut berada di area yang jauh lebih sparse dibanding tetangganya.

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    n_jobs=-1
)

df['LOF_Pred'] = lof.fit_predict(X_scaled)
df['LOF_Score'] = lof.negative_outlier_factor_

n_anomalies_lof = (df['LOF_Pred'] == -1).sum()
print(f'LOF detected {n_anomalies_lof} anomalies ({n_anomalies_lof/len(df)*100:.1f}%)')

In [ ]:
# perbandingan hasil kedua model
both_anomaly = ((df['IsoForest_Pred'] == -1) & (df['LOF_Pred'] == -1)).sum()
iso_only = ((df['IsoForest_Pred'] == -1) & (df['LOF_Pred'] == 1)).sum()
lof_only = ((df['IsoForest_Pred'] == 1) & (df['LOF_Pred'] == -1)).sum()

print(f'Flagged oleh kedua model: {both_anomaly}')
print(f'Hanya Isolation Forest: {iso_only}')
print(f'Hanya LOF: {lof_only}')
print(f'\nTotal unique anomalies (union): {both_anomaly + iso_only + lof_only}')

In [ ]:
# consensus: transaksi yang di-flag oleh kedua model lebih likely anomali
df['Consensus_Anomaly'] = ((df['IsoForest_Pred'] == -1) & (df['LOF_Pred'] == -1)).astype(int)

print(f'Consensus anomalies: {df["Consensus_Anomaly"].sum()}')

## 4. Analisis Karakteristik Anomali

Kita bandingkan profil transaksi normal vs anomali untuk validasi apakah model menangkap pola yang masuk akal secara bisnis.

In [ ]:
# profil anomali vs normal (berdasarkan Isolation Forest)
anomaly_df = df[df['IsoForest_Pred'] == -1]
normal_df = df[df['IsoForest_Pred'] == 1]

compare_cols = ['TransactionAmount', 'AccountBalance', 'TransactionDuration', 
                'LoginAttempts', 'CustomerAge', 'AmountToBalanceRatio', 'DaysSincePrevTxn']

comparison = pd.DataFrame({
    'Normal_Mean': normal_df[compare_cols].mean(),
    'Anomaly_Mean': anomaly_df[compare_cols].mean(),
    'Normal_Median': normal_df[compare_cols].median(),
    'Anomaly_Median': anomaly_df[compare_cols].median()
}).round(2)

comparison

In [ ]:
# distribusi channel dan transaction type pada anomali
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# channel distribution
channel_normal = normal_df['Channel'].value_counts(normalize=True)
channel_anomaly = anomaly_df['Channel'].value_counts(normalize=True)
channel_comp = pd.DataFrame({'Normal': channel_normal, 'Anomaly': channel_anomaly}).fillna(0)
channel_comp.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('Channel Distribution: Normal vs Anomaly')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=0)

# transaction type distribution
type_normal = normal_df['TransactionType'].value_counts(normalize=True)
type_anomaly = anomaly_df['TransactionType'].value_counts(normalize=True)
type_comp = pd.DataFrame({'Normal': type_normal, 'Anomaly': type_anomaly}).fillna(0)
type_comp.plot(kind='bar', ax=axes[1], edgecolor='black')
axes[1].set_title('Transaction Type: Normal vs Anomaly')
axes[1].set_ylabel('Proportion')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# feature importance: mana fitur yang paling berpengaruh terhadap anomaly score
# kita hitung korelasi antara tiap fitur dengan anomaly score
score_corr = X_scaled.copy()
score_corr['AnomalyScore'] = df['IsoForest_Score']
feat_importance = score_corr.corr()['AnomalyScore'].drop('AnomalyScore').sort_values()

plt.figure(figsize=(10, 6))
feat_importance.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Korelasi Fitur dengan Anomaly Score (Isolation Forest)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

print('Fitur dengan korelasi negatif kuat = semakin tinggi nilainya, semakin anomalous')

## 5. Thresholding

Selain menggunakan contamination rate tetap, kita juga bisa mengatur threshold berdasarkan anomaly score untuk mengontrol sensitivity.

In [ ]:
# distribusi score dan berbagai threshold
percentiles = [1, 3, 5, 10]

print('Threshold analysis berdasarkan percentile anomaly score:')
print('-' * 60)
for p in percentiles:
    threshold = np.percentile(df['IsoForest_Score'], p)
    n_flagged = (df['IsoForest_Score'] <= threshold).sum()
    print(f'  Percentile {p}%: threshold={threshold:.4f}, flagged={n_flagged} transaksi ({n_flagged/len(df)*100:.1f}%)')

print('\nDengan contamination=0.05, model secara default menggunakan ~5th percentile sebagai cutoff.')

## 6. Save Model

Simpan model dan scaler untuk dipakai di API inference nanti.

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(iso_forest, '../models/isolation_forest.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(le_channel, '../models/le_channel.joblib')
joblib.dump(le_txntype, '../models/le_txntype.joblib')
joblib.dump(le_occupation, '../models/le_occupation.joblib')
joblib.dump(feature_cols, '../models/feature_cols.joblib')

print('Model artifacts saved to ../models/')
print(f'  - isolation_forest.joblib')
print(f'  - scaler.joblib')
print(f'  - le_channel.joblib, le_txntype.joblib, le_occupation.joblib')
print(f'  - feature_cols.joblib')

## 7. Kesimpulan

**Model yang dipilih:** Isolation Forest dengan contamination rate 5%.

**Alasan pemilihan:**
- Cocok untuk unsupervised anomaly detection tanpa label
- Scalable dan efisien untuk dataset transaksional
- Hasil validasi kualitatif menunjukkan anomali yang terdeteksi memiliki karakteristik yang masuk akal (amount tinggi, login attempts banyak, dll)

**Feature engineering:**
- Temporal features: jam transaksi, hari, jarak antar transaksi
- Ratio features: amount/balance ratio
- Behavioral flags: late night transaction
- Encoded categoricals: channel, transaction type, occupation

**Evaluasi:**
- Karena tidak ada ground truth label, evaluasi dilakukan secara kualitatif dengan membandingkan profil anomali vs normal
- Cross-validation dengan LOF menunjukkan ada overlap yang reasonable antara kedua model
- Threshold bisa disesuaikan tergantung appetite risiko bisnis (lebih ketat = lebih banyak false positive, lebih longgar = lebih banyak false negative)